# SQL 기초 — 테이블 결합과 중첩 쿼리  (SQLite 노트북)
원본 파일: `w5_SQL_기초_테이블_결합과_중첩_쿼리.sql` · SQL 셀 34개

## 사용법
1. **위에서부터 순서대로** 실행하세요. 앞쪽 셀이 테이블(`customers`, `orders` 등)을 만듭니다. 데이터 생성 셀은 `DROP TABLE IF EXISTS`로 시작해서 다시 실행해도 안전합니다.
2. 연습 문제 셀은 주석 아래에 SQL을 쓰고 실행합니다. 원본 파일에 이미 풀이가 들어 있던 셀은 그대로 가져왔습니다.
3. 준비물: Python 3, `pandas` (`pip install pandas`), VS Code의 Python·Jupyter 확장. `sqlite3`는 파이썬에 내장되어 있어서 설치가 필요 없습니다.

## 원본(BigQuery/DuckDB)에서 자동 변환한 내용
- 테이블 경로 `` `aiffel-515805.modumall.x` ``, `project_name.dataset_name.x` → `x`
- `CREATE OR REPLACE TABLE` → `DROP TABLE IF EXISTS` + `CREATE TABLE`, 열 타입 `STRING`/`INT64`/`DECIMAL`/`DATE` → `TEXT`/`INTEGER`/`REAL`/`TEXT`
- 날짜는 SQLite에 별도 타입이 없어서 `'YYYY-MM-DD'` 텍스트로 저장합니다.

### 셀별 추가 변환
- **C9**: 원본의 끝 마침표(.) 오타 제거
- **C14**: 원본의 끝 마침표(.) 오타 제거
- **C17**: 원본의 끝 마침표(.) 오타 제거 / UNION DISTINCT → UNION
- **C18**: UNION DISTINCT → UNION
- **C33**: DATE_TRUNC → date(열, 'start of month')

> 검증: 원본 SQL을 DuckDB에서, 변환본을 SQLite에서 각각 실행해 셀별 결과를 비교했습니다. 값이 다른 셀은 없었습니다. (정렬 기준이 없는 쿼리는 행 순서가, 열 이름을 직접 안 붙인 `count(*)` 같은 열은 자동 이름이 엔진마다 다를 수 있습니다.)

In [1]:
import re, sqlite3
import pandas as pd
try:
    from IPython.display import display
except ImportError:          # 일반 파이썬에서 실행할 때
    display = print

con = sqlite3.connect(":memory:")   # 메모리 DB: 노트북(커널)을 끄면 사라집니다

def sql(text):
    """SQL을 실행합니다. SELECT류는 표로 보여주고, CREATE/INSERT 등은 실행만 합니다."""
    stmts, buf = [], ""
    for line in text.splitlines(keepends=True):
        buf += line
        if sqlite3.complete_statement(buf):      # ; 로 끝나는 문장 단위로 분리
            stmts.append(buf.strip()); buf = ""
    if buf.strip():
        stmts.append(buf.strip())
    ran = shown = 0
    for st in stmts:
        body = re.sub(r"--[^\n]*|/\*.*?\*/", "", st, flags=re.S).strip().rstrip(";").strip()
        if not body:
            continue                              # 주석만 있는 조각은 건너뜀
        ran += 1
        if body.split(None, 1)[0].upper() in ("SELECT", "WITH", "PRAGMA", "VALUES"):
            display(pd.read_sql_query(st, con)); shown += 1
        else:
            con.execute(st)
    con.commit()
    if ran == 0:
        print("(아직 작성한 SQL이 없습니다. 주석 아래에 쿼리를 작성한 뒤 다시 실행하세요.)")
    elif shown == 0:
        print("실행 완료")

print("SQLite 버전:", sqlite3.sqlite_version)

SQLite 버전: 3.53.4


## 1. 두 환경의 문법 차이 — 한눈에 보기

> **[C1] 건너뜀** — 원본의 DuckDB/BigQuery용 이름 공간 준비 셀입니다. SQLite는 필요 없습니다.

In [2]:
sql(r"""
-- [C2]
DROP TABLE IF EXISTS customers;
CREATE TABLE customers (
  customer_id TEXT,
  name TEXT,
  country TEXT,
  signup_date TEXT,
  grade TEXT
);

INSERT INTO customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze')
""")

실행 완료


In [3]:
sql(r"""
-- [C3]
DROP TABLE IF EXISTS products;
CREATE TABLE products (
  product_id TEXT,
  product_name TEXT,
  category TEXT,
  price REAL
);

INSERT INTO products VALUES
  ('P01', '에어러너', 'Running', 89000),
  ('P02', '클래식 스니커즈', 'Sneakers', 65000),
  ('P03', '첼시 부츠', 'Boots', 145000),
  ('P04', '여름 샌들', 'Sandals', 38000),
  ('P05', '트레일 러너', 'Running', 119000),
  ('P06', '캔버스 스니커즈', 'Sneakers', 49000),
  ('P07', '워커 부츠', 'Boots', 175000),
  ('P08', '슬리퍼 샌들', 'Sandals', 25000),
  ('P09', '양말 세트', 'Accessory', 12000),
  ('P10', '운동화 끈', 'Accessory', 5000)
""")

실행 완료


In [4]:
sql(r"""
-- [C4]
DROP TABLE IF EXISTS orders;
CREATE TABLE orders (
  order_id TEXT,
  customer_id TEXT,
  order_date TEXT,
  status TEXT,
  amount REAL
);

INSERT INTO orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000)
""")

실행 완료


In [5]:
sql(r"""
-- [C5]
DROP TABLE IF EXISTS order_items;
CREATE TABLE order_items (
  order_id TEXT,
  product_id TEXT,
  quantity INTEGER,
  unit_price REAL
);

INSERT INTO order_items VALUES
  ('O0001', 'P01', 1, 125000),
  ('O0002', 'P02', 2, 44500),
  ('O0003', 'P03', 1, 45000),
  ('O0004', 'P04', 2, 115000),
  ('O0006', 'P05', 1, 67000),
  ('O0007', 'P06', 2, 79000),
  ('O0008', 'P07', 1, 32000),
  ('O0009', 'P08', 2, 205000),
  ('O0010', 'P09', 1, 99000),
  ('O0011', 'P01', 2, 38000),
  ('O0013', 'P02', 1, 142000),
  ('O0014', 'P03', 2, 44000),
  ('O0015', 'P04', 1, 53000),
  ('O0016', 'P05', 2, 87500),
  ('O0017', 'P06', 1, 61000),
  ('O0018', 'P07', 2, 160000),
  ('O0019', 'P08', 1, 47000),
  ('O0020', 'P09', 2, 107500),
  ('O0021', 'P01', 1, 38000),
  ('O0022', 'P02', 2, 67000),
  ('O0023', 'P03', 1, 92000),
  ('O0024', 'P04', 2, 134000),
  ('O0026', 'P05', 1, 119000),
  ('O0027', 'P06', 2, 202500),
  ('O0028', 'P07', 1, 58000),
  ('O0029', 'P08', 2, 36500),
  ('O0030', 'P09', 1, 187000)
""")

실행 완료


## 2. ⌨️ 백문이 불여일타 (1)

In [6]:
sql(r"""
-- [C6] ⌨️ 백문이 불여일타 (1) — customer_id의 PK 자격(유일성) 확인

select 
  count(*), 
  count(distinct customer_id)
from customers
""")

,count(*),count(distinct customer_id)
0,15,15


### 3. ▶️ **코드 실행하기 · 코드 셀 7 [C7]**

In [7]:
sql(r"""
-- [C7]
SELECT
  o.order_id,
  o.amount,
  c.name AS 고객명,
  c.country AS 국가
FROM orders AS o
JOIN customers AS c
  ON o.customer_id = c.customer_id
WHERE
  o.amount IS NOT NULL
ORDER BY
  o.amount DESC
LIMIT 5
""")

,order_id,amount,고객명,국가
0,O0009,410000.0,조은우,Japan
1,O0027,405000.0,권준서,Korea
2,O0018,320000.0,권준서,Korea
3,O0024,268000.0,조은우,Japan
4,O0004,230000.0,박도윤,Japan


### 4. ▶️ **코드 실행하기 · 코드 셀 8 [C8]**

In [8]:
sql(r"""
-- [C8]
SELECT
  oi.order_id,
  p.product_name AS 상품명,
  p.category AS 카테고리,
  oi.quantity AS 수량,
  oi.unit_price AS 단가,
  oi.quantity * oi.unit_price AS 라인합계
FROM order_items AS oi
JOIN products AS p
  ON oi.product_id = p.product_id
ORDER BY
  라인합계 DESC
LIMIT 5
""")

,order_id,상품명,카테고리,수량,단가,라인합계
0,O0009,슬리퍼 샌들,Sandals,2,205000.0,410000.0
1,O0027,캔버스 스니커즈,Sneakers,2,202500.0,405000.0
2,O0018,워커 부츠,Boots,2,160000.0,320000.0
3,O0024,여름 샌들,Sandals,2,134000.0,268000.0
4,O0004,여름 샌들,Sandals,2,115000.0,230000.0


## 5. ⌨️ 백문이 불여일타 (2)

In [9]:
sql(r"""
-- [C9] ⌨️ 백문이 불여일타 (2) — 주문번호·금액·고객 등급 조회

select 
  o.order_id, 
  o.amount, 
  c.customer_id,
  c.name,
  c.grade
from orders o
join customers c
  on o.customer_id = c.customer_id
order by amount desc
limit 5
""")

,order_id,amount,customer_id,name,grade
0,O0009,410000.0,C007,조은우,Silver
1,O0027,405000.0,C013,권준서,Gold
2,O0018,320000.0,C013,권준서,Gold
3,O0024,268000.0,C007,조은우,Silver
4,O0004,230000.0,C003,박도윤,Bronze


## 6. ⌨️ 백문이 불여일타 (3)

In [10]:
sql(r"""
-- [C10] ⌨️ 백문이 불여일타 (3) — 카테고리별 총 판매수량

select p.category, sum(oi.quantity * oi.unit_price) as total
from order_items oi
join products p
  on oi.product_id = p.product_id
group by p.category
order by sum(oi.quantity * oi.unit_price) desc
""")

,category,total
0,Sandals,1081000.0
1,Sneakers,989000.0
2,Boots,635000.0
3,Running,600000.0
4,Accessory,501000.0


### 7. ▶️ **코드 실행하기 · 코드 셀 11 [C11]**

In [11]:
sql(r"""
-- [C11]
SELECT
  o.order_id,
  o.amount,
  oi.product_id,
  oi.quantity
FROM orders AS o
JOIN order_items AS oi
  ON o.order_id = oi.order_id
ORDER BY
  o.order_id NULLS LAST
LIMIT 6
""")

,order_id,amount,product_id,quantity
0,O0001,125000.0,P01,1
1,O0002,89000.0,P02,2
2,O0003,45000.0,P03,1
3,O0004,230000.0,P04,2
4,O0006,67000.0,P05,1
5,O0007,158000.0,P06,2


## 8. ⌨️ 백문이 불여일타 (4)

In [12]:
sql(r"""
-- [C12] ⌨️ 백문이 불여일타 (4) — 국가별 매출 구하기

select c.country, sum(o.amount) as total_amount
from orders o
join customers c
  on o.customer_id = c.customer_id
where o.amount is not null and c.country is not null
group by c.country
order by sum(o.amount) desc
""")

,country,total_amount
0,Korea,1851000.0
1,Japan,1389000.0
2,USA,513000.0


### 9. ▶️ **코드 실행하기 · 코드 셀 13 [C13]**

In [13]:
sql(r"""
-- [C13]
SELECT
  p.product_id,
  p.product_name AS 상품명,
  SUM(oi.quantity) AS 총판매수량
FROM products AS p
LEFT JOIN order_items AS oi
  ON p.product_id = oi.product_id
GROUP BY
  p.product_id,
  상품명
ORDER BY
  총판매수량 NULLS LAST
""")

,product_id,상품명,총판매수량
0,P01,에어러너,4.0
1,P03,첼시 부츠,4.0
2,P05,트레일 러너,4.0
3,P07,워커 부츠,4.0
4,P09,양말 세트,4.0
5,P02,클래식 스니커즈,5.0
6,P04,여름 샌들,5.0
7,P06,캔버스 스니커즈,5.0
8,P08,슬리퍼 샌들,5.0
9,P10,운동화 끈,NaN


## 10. ⌨️ 백문이 불여일타 (5)

In [14]:
sql(r"""
-- [C14] ⌨️ 백문이 불여일타 (5) — 한 번도 팔리지 않은 상품 찾기

/*SELECT
  p.product_id AS 제품번호,
  p.product_name AS 상품명,
  SUM(oi.quantity) AS 총판매수량
FROM products AS p
LEFT JOIN order_items AS oi
  ON p.product_id = oi.product_id
GROUP BY
  p.product_id,
  p.product_name
having sum(oi.quantity) is null*/

SELECT
  p.product_id,
  p.product_name
FROM products AS p
LEFT JOIN order_items AS oi
  ON p.product_id = oi.product_id
WHERE
  oi.order_id IS NULL
""")

,product_id,product_name
0,P10,운동화 끈


### 11. ▶️ **코드 실행하기 · 코드 셀 15 [C15]**

In [15]:
sql(r"""
-- [C15]
SELECT
  p.product_name AS 상품명,
  SUM(oi.quantity) AS 총판매수량
FROM order_items AS oi
RIGHT JOIN products AS p
  ON oi.product_id = p.product_id
GROUP BY
  상품명
ORDER BY
  총판매수량 NULLS LAST
""")

,상품명,총판매수량
0,양말 세트,4.0
1,에어러너,4.0
2,워커 부츠,4.0
3,첼시 부츠,4.0
4,트레일 러너,4.0
5,슬리퍼 샌들,5.0
6,여름 샌들,5.0
7,캔버스 스니커즈,5.0
8,클래식 스니커즈,5.0
9,운동화 끈,NaN


### 12. ▶️ **코드 실행하기 · 코드 셀 16 [C16]**

In [16]:
sql(r"""
-- [C16]
SELECT
  name,
  '골드회원' AS 사유
FROM customers
WHERE
  grade = 'Gold'
UNION ALL
SELECT
  name,
  '한국고객' AS 사유
FROM customers
WHERE
  country = 'Korea'
ORDER BY
  name NULLS LAST
""")

,name,사유
0,강서윤,한국고객
1,권준서,골드회원
2,권준서,한국고객
3,김민준,골드회원
4,김민준,한국고객
5,안수아,한국고객
6,윤지호,골드회원
7,이서연,한국고객
8,임하은,한국고객
9,정하준,한국고객


## 13. ⌨️ 백문이 불여일타 (6)

In [17]:
sql(r"""
-- [C17] ⌨️ 백문이 불여일타 (6) — UNION으로 주의 주문 목록 만들기

SELECT o.order_id AS 주의주문
FROM orders AS o
WHERE o.amount >= 300000

UNION

SELECT o.order_id
FROM orders AS o
WHERE o.status = 'Returned'
""")

,주의주문
0,O0003
1,O0009
2,O0017
3,O0018
4,O0027
5,O0029


### 14. ▶️ **코드 실행하기 · 코드 셀 18 [C18]**

In [18]:
sql(r"""
-- [C18]
SELECT
  'UNION DISTINCT (중복제거)' AS 방식,
  COUNT(*) AS 행수
FROM (
  SELECT
    name
  FROM customers
  WHERE
    grade = 'Gold'
  UNION
  SELECT
    name
  FROM customers
  WHERE
    country = 'Korea'
) AS t
UNION ALL
SELECT
  'UNION ALL (중복유지)',
  COUNT(*)
FROM (
  SELECT
    name
  FROM customers
  WHERE
    grade = 'Gold'
  UNION ALL
  SELECT
    name
  FROM customers
  WHERE
    country = 'Korea'
) AS t
""")

,방식,행수
0,UNION DISTINCT (중복제거),10
1,UNION ALL (중복유지),12


### 15. ▶️ **코드 실행하기 · 코드 셀 19 [C19]**

In [19]:
sql(r"""
-- [C19]
SELECT
  order_id,
  amount
FROM orders
WHERE
  amount > (
    SELECT
      AVG(amount)
    FROM orders
  )
ORDER BY
  amount DESC
""")

,order_id,amount
0,O0009,410000.0
1,O0027,405000.0
2,O0018,320000.0
3,O0024,268000.0
4,O0004,230000.0
5,O0020,215000.0
6,O0030,187000.0
7,O0016,175000.0
8,O0007,158000.0
9,O0013,142000.0


### 16. ▶️ **코드 실행하기 · 코드 셀 20 [C20]**

In [20]:
sql(r"""
-- [C20]
SELECT
  order_id,
  customer_id,
  amount
FROM orders
WHERE
  customer_id IN (
    SELECT
      customer_id
    FROM customers
    WHERE
      country = 'USA'
  )
""")

,order_id,customer_id,amount
0,O0005,C004,NaN
1,O0010,C008,99000.0
2,O0014,C004,88000.0
3,O0020,C008,215000.0
4,O0021,C014,38000.0
5,O0029,C004,73000.0


### 17. ▶️ **코드 실행하기 · 코드 셀 21 [C21]**

In [21]:
sql(r"""
-- [C21]
SELECT
  *
FROM (
  SELECT
    customer_id,
    SUM(amount) AS total_amount
  FROM orders
  WHERE
    amount IS NOT NULL
  GROUP BY
    customer_id
) AS customer_total
WHERE
  total_amount >= 200000
ORDER BY
  total_amount DESC
""")

,customer_id,total_amount
0,C013,725000.0
1,C007,678000.0
2,C001,380000.0
3,C012,362000.0
4,C003,349000.0
5,C008,314000.0
6,C002,308000.0


## 18. ⌨️ 백문이 불여일타 (7)

In [22]:
sql(r"""
-- [C22] ⌨️ 백문이 불여일타 (7) — 서브쿼리로 한국 고객 주문 조회

select order_id, amount
from orders
where customer_id in(
  select customer_id
  from customers
  where country = 'USA'
)
  and amount is not null
order by amount desc
""")

,order_id,amount
0,O0020,215000.0
1,O0010,99000.0
2,O0014,88000.0
3,O0029,73000.0
4,O0021,38000.0


### 19. ▶️ **코드 실행하기 · 코드 셀 23 [C23]**

In [23]:
sql(r"""
-- [C23]
SELECT
  order_id,
  amount,
  (
    SELECT
      ROUND(AVG(amount), 0)
    FROM orders
  ) AS 전체평균,
  amount - (
    SELECT
      AVG(amount)
    FROM orders
  ) AS 평균과의차이
FROM orders
WHERE
  amount IS NOT NULL
ORDER BY
  amount DESC
LIMIT 5
""")

,order_id,amount,전체평균,평균과의차이
0,O0009,410000.0,140963.0,269037.037037
1,O0027,405000.0,140963.0,264037.037037
2,O0018,320000.0,140963.0,179037.037037
3,O0024,268000.0,140963.0,127037.037037
4,O0004,230000.0,140963.0,89037.037037


### 20. ▶️ **코드 실행하기 · 코드 셀 24 [C24]**

In [24]:
sql(r"""
-- [C24]
SELECT
  c.country,
  p.category,
  SUM(oi.quantity * oi.unit_price) AS 매출
FROM customers AS c
JOIN orders AS o
  ON c.customer_id = o.customer_id
JOIN order_items AS oi
  ON o.order_id = oi.order_id
JOIN products AS p
  ON oi.product_id = p.product_id
WHERE
  c.country IS NOT NULL
GROUP BY
  c.country,
  p.category
ORDER BY
  매출 DESC
""")

,country,category,매출
0,Korea,Sneakers,989000.0
1,Japan,Sandals,908000.0
2,Korea,Boots,547000.0
3,USA,Accessory,314000.0
4,Japan,Running,294000.0
5,Korea,Running,268000.0
6,Japan,Accessory,187000.0
7,USA,Boots,88000.0
8,USA,Sandals,73000.0
9,Korea,Sandals,47000.0


### 21. ▶️ **코드 실행하기 · 코드 셀 25 [C25]**

In [25]:
sql(r"""
-- [C25]
SELECT
  customer_id,
  name,
  country
FROM customers AS c
WHERE
  EXISTS (
    SELECT 1
    FROM orders AS o
    WHERE o.customer_id = c.customer_id
  )
ORDER BY
  customer_id
""")

,customer_id,name,country
0,C001,김민준,Korea
1,C002,이서연,Korea
2,C003,박도윤,Japan
3,C004,최지우,USA
4,C005,정하준,Korea
5,C006,강서윤,Korea
6,C007,조은우,Japan
7,C008,윤지호,USA
8,C009,임하은,Korea
9,C010,한예준,Korea


## 22. 📒 연습 문제 — 조인과 서브쿼리 (5문항)

In [26]:
sql(r"""
-- [C26] 📒 연습 문제 — 조인과 서브쿼리 (1/5)

-- 문제: 고객의 국가(country)별 주문 건수를 구하고, 많은 순으로 정렬합니다.

select c.country, count(*) as 주문건수
from customers c
join orders o
  on o.customer_id = c.customer_id
where c.country is not null
  group by country
order by 주문건수 desc
""")

,country,주문건수
0,Korea,17
1,USA,6
2,Japan,6


In [27]:
sql(r"""
-- [C27] 📒 연습 문제 — 조인과 서브쿼리 (2/5)

-- 문제: orders·order_items·products 세 테이블을 조인해 주문번호·상품명·수량·단가를 조회합니다.

select 
  o.order_id, 
  p.product_name, 
  oi.quantity, 
  oi.unit_price
FROM orders AS o
join order_items oi
  on o.order_id = oi.order_id
join products p
  on oi.product_id = p.product_id
""")

,order_id,product_name,quantity,unit_price
0,O0001,에어러너,1,125000.0
1,O0002,클래식 스니커즈,2,44500.0
2,O0003,첼시 부츠,1,45000.0
3,O0004,여름 샌들,2,115000.0
4,O0006,트레일 러너,1,67000.0
5,O0007,캔버스 스니커즈,2,79000.0
6,O0008,워커 부츠,1,32000.0
7,O0009,슬리퍼 샌들,2,205000.0
8,O0010,양말 세트,1,99000.0
9,O0011,에어러너,2,38000.0


In [28]:
sql(r"""
-- [C28] 📒 연습 문제 — 조인과 서브쿼리 (3/5)

-- 문제: 주문이 한 건도 없는 고객의 이름과 국가를 찾으세요. (LEFT JOIN 활용)

select c.customer_id, c.country
from customers c
left join orders o
  on c.customer_id = o.customer_id
where o.amount is null
""")

,customer_id,country
0,C004,USA
1,C009,Korea
2,C010,Korea


In [29]:
sql(r"""
-- [C29] 📒 연습 문제 — 조인과 서브쿼리 (4/5)

-- 문제: 카테고리별 총매출(수량 × 단가)을 구하고 매출이 큰 순으로 정렬합니다.

select p.category, sum(oi.quantity * oi.unit_price) as 총매출
from products p
join order_items oi
  on p.product_id = oi.product_id
group by p.category
order by 총매출 desc
""")

,category,총매출
0,Sandals,1081000.0
1,Sneakers,989000.0
2,Boots,635000.0
3,Running,600000.0
4,Accessory,501000.0


In [30]:
sql(r"""
-- [C30] 📒 연습 문제 — 조인과 서브쿼리 (5/5)

-- 문제: 전체 주문의 평균 금액보다 큰 주문만 조회합니다. (서브쿼리 활용)

select order_id, amount
from orders
where amount > (select avg(amount)
  from orders 
)
""")

,order_id,amount
0,O0004,230000.0
1,O0007,158000.0
2,O0009,410000.0
3,O0013,142000.0
4,O0016,175000.0
5,O0018,320000.0
6,O0020,215000.0
7,O0024,268000.0
8,O0027,405000.0
9,O0030,187000.0


## 27. 🧪 종합 실습 — 월별·카테고리별 매출 TOP 10

In [31]:
sql(r"""
-- [C31] 🧪 종합 실습 — 월별·카테고리별 매출 TOP 10 (1/4)

select p.category, sum(oi.quantity * oi.unit_price)
from products p
join order_items oi
  on p.product_id = oi.product_id
group by p.category
order by sum(oi.quantity * oi.unit_price) desc
""")

,category,sum(oi.quantity * oi.unit_price)
0,Sandals,1081000.0
1,Sneakers,989000.0
2,Boots,635000.0
3,Running,600000.0
4,Accessory,501000.0


## 28. **질문 2.** 고객 이름별 총 구매액 TOP 5를 구합니다. (`orders` + `customers`)

In [32]:
sql(r"""
-- [C32] 🧪 종합 실습 — 월별·카테고리별 매출 TOP 10 (2/4)

SELECT
  c.name AS 고객명,
  c.grade AS 등급,
  SUM(o.amount) AS 총구매액
FROM orders AS o
JOIN customers AS c
  ON o.customer_id = c.customer_id
WHERE
  o.amount IS NOT NULL
GROUP BY
  고객명,
  등급
ORDER BY
  총구매액 DESC
LIMIT 5
""")

,고객명,등급,총구매액
0,권준서,Gold,725000.0
1,조은우,Silver,678000.0
2,김민준,Gold,380000.0
3,신아린,Silver,362000.0
4,박도윤,Bronze,349000.0


## 29. **질문 3.** 월별 × 카테고리별 매출을 구해, 매출 상위 10개 (월, 카테고리) 조합을 보여주세요. 그리고 카테고리별 매출을 그래프로 그립니다.

In [33]:
sql(r"""
-- [C33] 🧪 종합 실습 — 월별·카테고리별 매출 TOP 10 (3/4)

SELECT
  date(o.order_date, 'start of month') AS 월,
  p.category AS 카테고리,
  SUM(oi.quantity * oi.unit_price) AS 매출
FROM orders AS o
JOIN order_items AS oi
  ON o.order_id = oi.order_id
JOIN products AS p
  ON oi.product_id = p.product_id
GROUP BY
  월,
  카테고리
ORDER BY
  매출 DESC
LIMIT 10
""")

,월,카테고리,매출
0,2023-10-01,Sandals,410000.0
1,2023-11-01,Boots,408000.0
2,2024-01-01,Sneakers,405000.0
3,2024-01-01,Sandals,268000.0
4,2023-09-01,Sandals,230000.0
5,2023-12-01,Accessory,215000.0
6,2023-11-01,Sneakers,203000.0
7,2023-09-01,Running,192000.0
8,2024-02-01,Accessory,187000.0
9,2023-11-01,Running,175000.0


In [34]:
sql(r"""
-- [C34] 🧪 종합 실습 — 월별·카테고리별 매출 TOP 10 (4/4)

SELECT
  p.category AS category,
  SUM(oi.quantity * oi.unit_price) AS revenue
FROM order_items AS oi
JOIN products AS p
  ON oi.product_id = p.product_id
GROUP BY
  category
ORDER BY
  revenue DESC
""")

,category,revenue
0,Sandals,1081000.0
1,Sneakers,989000.0
2,Boots,635000.0
3,Running,600000.0
4,Accessory,501000.0


## 31. 코드 퀴즈

In [35]:
sql(r"""
-- [C35] 코드 퀴즈 — 등급별 총 매출

SELECT
  c.grade,
  SUM(o.amount) AS 총매출
FROM orders AS o
JOIN customers AS c
  ON o.customer_id = c.customer_id
WHERE
  o.amount IS NOT NULL
GROUP BY
  c.grade
ORDER BY
  총매출 DESC
""")

,grade,총매출
0,Silver,1604000.0
1,Gold,1580000.0
2,Bronze,584000.0
3,NaN,38000.0
